In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# ----------------------------------------------------
# 1. 全局超参数定义 (在真实架构中，这些通常写进 Config 类)
# ----------------------------------------------------
batch_size = 32    # B: 并行处理的序列数量
block_size = 8     # T: 序列的最大上下文长度 
n_embd = 32        # C: 每一个 token 被映射到的特征维度 

head_size = 16     # 单个注意力头投影后的特征维度 

torch.manual_seed(1337) # 保证每次输出一样，方便调试

# ----------------------------------------------------
# 2. 模拟底层的输入数据流动
# ----------------------------------------------------
x = torch.randn(batch_size, block_size, n_embd)

class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        
        self.key = nn.Linear(n_embd,head_size,bias=False)#Wk,Wq形状都是[n_embd,head_size]
        self.query = nn.Linear(n_embd,head_size,bias=False)
        self.value = nn.Linear(n_embd,head_size,bias=False)

        # register_buffer 用于注册一个不需要梯度更新的持久化状态（即掩码矩阵）
        # 我们必须按照模型能容纳的“最大上下文长度 (block_size)”来申请这个全 1 的下三角矩阵。
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self,x):
        B,T,C = x.shape

        #计算Q，K
        k = self.key(x) #执行的是高维的矩阵乘法k=x*Wk
        q = self.query(x)

        #计算注意力分数（包含缩放点积注意力）
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5#输出矩阵[T,T]

        #因果掩码
        # 动态切片截取：只从预先分配好的大矩阵 self.tril 中，切出当前句子实际长度的 [:T, :T] 块
        wei = wei.masked_fill(self.tril[:T,:T] == 0,float('-inf'))

        #归一化为概率分布
        wei = F.softmax(wei,dim=-1)#形状[B,T,T]

        #信息聚合
        v = self.value(x)
        out = wei @ v

        return out

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

        #最后的线性投影层
        self.proj = nn.Linear(n_embd,n_embd)

    def forward(self,x):
        #1、并行执行所有注意力头
        head_outputs = [h(x) for h in self.heads]

        #2、在特征维度上进行拼接
        out = torch.cat(head_outputs,dim = -1)

        #3、经过一次线性投影，混合特征
        out = self.proj(out)

        return out

In [ ]:
class FeedForward(nn.Module):

    def __init__(self,n_embd):
        super.__init__()

        # 按照 Transformer 的标准架构，隐藏层的维度通常会放大 4 倍
        self.net = nn.Sequential(
            nn.Linear(n_embd,4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd,n_embd)#最后的投影层
        )

    def forward(self,x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self,n_embd,n_head):
        super.__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(num_heads=n_head,head_size=head_size)
        self.ffwd = FeedForward(n_embd)
 
        #Layer Normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        # 这里的 x = x + ... 就是残差连接 (Residual Connection)
        # 注意：现代 Transformer (如 GPT-2) 普遍采用 Pre-Norm 架构，
        # 即在进入 Attention 和 FeedForward 之前先做 LayerNorm。

        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
